# W04-01 · 문서와 청킹

필수 20분 + 확장 15분. 원본과 조각의 관계, 겹치는 구간, 문서 유효기간을 직접 확인합니다.

**진행 방식**: 먼저 실행 결과를 예측하고 셀을 실행합니다. 변경 실습은 제공된 값 하나를 바꿉니다. 핵심 셀은 네트워크·API 키 없이 실행됩니다. 모든 정책·질의는 교육용 합성 데이터입니다.

VS Code → 우측 상단 Select Kernel → 이 저장소의 .venv (Python 3.11). 노트북별로 Restart 후 Run All 가능합니다.

In [ ]:
from pathlib import Path
import sys, json, math, statistics
from dataclasses import asdict, replace
from collections import Counter
from time import perf_counter

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course_labs").is_dir()), None)
assert ROOT is not None, "저장소 내부에서 notebook을 열어 주세요."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_labs.retrieval import (
    Document,
    Chunk,
    LexicalIndex,
    load_documents,
    load_cases,
    chunk_document,
    tokenize,
    expand_query,
    eligible_chunks,
    reciprocal_rank_fusion,
    rerank_by_coverage,
    retrieval_metrics,
    retrieve,
    answer_from_evidence,
    evaluate,
)
from IPython.display import display, HTML

docs = load_documents()
cases = load_cases()
print("Python:", sys.version.split()[0], "| 문서:", len(docs), "| 평가 질의:", len(cases))
assert sys.version_info[:2] == (3, 11), "과정 기준 Python 3.11.x 커널 선택"

In [ ]:
def show(rows, columns=None):
    import html

    if not rows:
        return display(HTML("<p>결과 없음</p>"))
    cols = columns or list(rows[0])
    headers = "".join(
        '<th style="padding:8px;text-align:left">' + html.escape(str(c)) + "</th>" for c in cols
    )
    body = "".join(
        "<tr>"
        + "".join(
            '<td style="padding:8px;border-bottom:1px solid #ddd">'
            + html.escape(str(r.get(c, "")))
            + "</td>"
            for c in cols
        )
        + "</tr>"
        for r in rows
    )
    display(
        HTML(
            '<table style="font-size:15px;border-collapse:collapse"><tr>'
            + headers
            + "</tr>"
            + body
            + "</table>"
        )
    )

## 01 문서 목록

검색 전에 자료의 출처와 사용 범위를 확인합니다.

In [ ]:
show(
    [
        {
            "id": d.id,
            "title": d.title,
            "version": d.version,
            "access": d.access,
            "from": d.valid_from,
            "to": d.valid_to,
        }
        for d in docs
    ]
)

**관찰·변경 과제**

D08과 D18이 오늘 기준 검색 대상인지 먼저 예측해 보세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 02 원문 위치

인용은 문서 이름뿐 아니라 해당 문장의 위치를 가리켜야 합니다.

In [ ]:
doc = docs[0]
print(doc.text)
chunks = chunk_document(doc, size=60, overlap=12)
show([asdict(c) for c in chunks], ["id", "start", "end", "text"])
assert all(doc.text[c.start : c.end] == c.text for c in chunks)

**관찰·변경 과제**

size를 45, 100으로 변경하여 조건과 예외가 함께 남는지 비교하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 03 겹침의 효과

겹침은 문맥 단절을 줄일 수 있지만 중복과 인덱스 크기를 늘립니다.

In [ ]:
rows = []
for overlap in [0, 12, 24, 36]:
    cs = chunk_document(doc, 60, overlap)
    rows.append(
        {
            "overlap": overlap,
            "chunks": len(cs),
            "stored_characters": sum(len(c.text) for c in cs),
            "original_characters": len(doc.text),
        }
    )
show(rows)

**관찰·변경 과제**

최대 겹침을 무조건 권장할 수 없는 이유를 저장 문자 수로 설명하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 04 문장 단위 분할

문자 창과 문장 경계의 차이를 관찰합니다.

In [ ]:
import re

sentences = [m.group(0) for m in re.finditer(r"[^.!?]+[.!?]?", doc.text)]
show(
    [{"sentence": i + 1, "text": s.strip(), "characters": len(s)} for i, s in enumerate(sentences)]
)

**관찰·변경 과제**

문장 하나가 너무 긴 정책에서 어떤 추가 규칙이 필요한가요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 05 청킹 위치 시각화

각 조각의 범위가 원본에서 어디에 있는지 그림으로 확인합니다.

In [ ]:
bars = "".join(
    f'<div style="margin:7px 0;margin-left:{c.start * 3}px;width:{len(c.text) * 3}px;background:#dce8ff;border-left:3px solid #235dcc;padding:5px;font-size:14px">{c.id} ({c.start}–{c.end})</div>'
    for c in chunks
)
display(HTML('<div style="width:650px;overflow:auto">' + bars + "</div>"))

**관찰·변경 과제**

overlap=0과 12의 그림을 각각 캡처해 차이를 설명하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 06 날짜 필터

valid_to는 제외 경계입니다. 시작일 포함, 종료일 제외를 명시합니다.

In [ ]:
all_chunks = [c for d in docs for c in chunk_document(d)]
for day in ["2026-08-31", "2026-09-01", "2026-09-29", "2027-01-01"]:
    ids = sorted({c.doc_id for c in eligible_chunks(all_chunks, day)})
    print(day, "D01?", "D01" in ids, "D08?", "D08" in ids, "D18?", "D18" in ids)

**관찰·변경 과제**

경계 날짜에 옛 정책과 새 정책이 동시에 포함되지 않는지 확인하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 07 권한 필터

권한 없는 자료는 검색 전에 제외합니다.

In [ ]:
public_ids = {c.doc_id for c in eligible_chunks(all_chunks, "2026-09-29", "public")}
staff_ids = {c.doc_id for c in eligible_chunks(all_chunks, "2026-09-29", "staff")}
print("staff 전용:", staff_ids - public_ids)
assert "D09" not in public_ids

**관찰·변경 과제**

역할 선택 UI는 실제 인증이 아닙니다. 서비스에서는 역할을 어느 신뢰 가능한 계층에서 결정해야 할까요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 08 잘못된 설정 처리

조각 크기와 겹침의 잘못된 입력은 명확하게 거절합니다.

In [ ]:
for size, overlap in [(60, 60), (60, -1), (0, 0)]:
    try:
        chunk_document(doc, size, overlap)
    except ValueError as e:
        print((size, overlap), str(e))

**관찰·변경 과제**

실패가 발생한 뒤 이전 검색 결과가 최신 결과처럼 보이지 않게 하려면?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 09 자기 문서 추가

개인정보 없는 합성 문서로 확장합니다.

In [ ]:
my_doc = Document(
    "MY01",
    "합성 상담 예약",
    "상담 예약은 평일 오후 2시부터 가능합니다. 예약 변경은 하루 전까지 신청합니다.",
)
my_chunks = chunk_document(my_doc, 60, 8)
show([asdict(c) for c in my_chunks], ["id", "text", "start", "end"])

**관찰·변경 과제**

본인 프로젝트의 공개 가능한 합성 규칙 3개로 바꿔 보세요. 실제 학생 연락처를 넣지 않습니다.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 10 실험 기록 보관

분할 규칙과 원본 해시를 함께 남깁니다.

In [ ]:
record = {"document": asdict(doc), "size": 60, "overlap": 12, "chunks": [asdict(c) for c in chunks]}
print(json.dumps(record, ensure_ascii=False, indent=2)[:800])

**관찰·변경 과제**

개인 작업 폴더에 JSON으로 저장하고, 같은 설정으로 재생성한 범위를 비교하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 완료 체크

- 설정 하나를 바꾼 비교 결과와 실패 질의 1개
- 출처·문서 버전·기준 날짜를 포함한 설명
- 측정한 것과 측정하지 않은 것의 구분

## 참고

- [RAG 원 논문](https://arxiv.org/abs/2005.11401)
- [Stanford IR: TF-IDF](https://nlp.stanford.edu/IR-book/html/htmledition/term-frequency-and-weighting-1.html)
- [Stanford IR: BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html)
- [Ollama embed API](https://docs.ollama.com/api/embed)

로컬 구현 위치: course_labs/retrieval.py. 형태소 분석·학습된 임베딩·신경망 재정렬을 구현한 것으로 해석하지 않습니다.